# MNLI analysis and tests

This notebook is a separate workspace for the slide-deck analysis. It imports the existing pipeline and does not edit the project source files.

Run `run_data.py` first for prepared-subset results, and `run_experiments.py` first for model metrics.

In [8]:
import json
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.data import audit_data, load_mnli, load_saved_folds
from pipeline.evaluation import evaluate_predictions

DATA_DIR = PROJECT_ROOT / "results" / "data" / "mnli-n50000-seed0"
EXPERIMENT_DIR = PROJECT_ROOT / "results" / "experiments" / "mnli-n50000-seed0"
OUTPUT_DIR = PROJECT_ROOT / "results" / "analysis"
LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}

def inspect_prepared_subset():
    if not (DATA_DIR / "manifest.json").exists():
        print("Prepared data is not available. Run `uv run python run_data.py` first.")
        return
    subset, folds = load_saved_folds(DATA_DIR)
    audit = audit_data(subset)
    print(f"Prepared subset size: {len(subset):,}")
    print("Labels:", {LABEL_NAMES.get(int(k), str(k)): v for k, v in audit["label_counts"].items()})
    for fold_id, fold in folds.items():
        print(f"Fold {fold_id}: {len(fold['training']):,} training, {len(fold['validation']):,} validation")

def inspect_experiment_metrics():
    if not EXPERIMENT_DIR.exists():
        print("Experiment results are not available. Run `uv run python run_experiments.py` first.")
        return
    rows = []
    for path in EXPERIMENT_DIR.glob("*/*/metrics.json"):
        metrics = json.loads(path.read_text(encoding="utf-8"))
        rows.append({"model": path.parent.parent.name, "fold": path.parent.name,
                     "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"]})
    if rows:
        table = pd.DataFrame(rows).sort_values(["model", "fold"])
        display(table)
        display(table.groupby("model")[["accuracy", "macro_f1"]].agg(["mean", "std"]).round(4))
    else:
        print("No metrics.json files found.")

def run_small_sanity_tests():
    assert set(LABEL_NAMES) == {0, 1, 2}
    print("Sanity tests passed.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Analysis output: {OUTPUT_DIR}")

Project root: /Users/lindevansaane/Onedrive/Msc Artificial Intelligence /Natural Language Processing 1/Projects/NLP1-Natural_Language_Inference-Project
Analysis output: /Users/lindevansaane/Onedrive/Msc Artificial Intelligence /Natural Language Processing 1/Projects/NLP1-Natural_Language_Inference-Project/results/analysis


## 1. Official MNLI dataset

In [9]:
# # mnli = load_mnli()

# # for split_name, split in mnli.items():
# #     print(f"{split_name}: {len(split):,} examples")

# from datasets import load_dataset

# mnli = {
#     "train": load_dataset(
#         "nyu-mll/multi_nli",
#         split="train[:5000]"
#     )
# }

# print(f"Loaded {len(mnli['train']):,} examples")

# import os

# os.environ["HF_HUB_DISABLE_XET"] = "1"
# os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# from datasets import Dataset, load_dataset

# stream = load_dataset(
#     "nyu-mll/multi_nli",
#     split="train",
#     streaming=True,
# )

# examples = list(stream.take(5000))
# mnli = {"train": Dataset.from_list(examples)}

# print(f"Loaded {len(mnli['train']):,} examples")

from datasets import load_dataset

mnli = load_dataset("nyu-mll/multi_nli")

for split_name, split in mnli.items():
    print(f"{split_name}: {len(split):,} examples")

train: 392,702 examples
validation_matched: 9,815 examples
validation_mismatched: 9,832 examples


In [10]:
for split_name, split in mnli.items():
    audit = audit_data(split)
    print(f"\n{split_name}")
    print("Label counts:", {
        LABEL_NAMES.get(int(label), str(label)): count
        for label, count in audit["label_counts"].items()
    })
    print("Genre counts:", audit["genre_counts"])
    print("Duplicate rows:", audit["duplicate_rows"])
    print("Conflicting pairs:", len(audit["conflicting_pairs"]))
    print("Word lengths:", audit["word_lengths"])


train
Label counts: {'neutral': 130900, 'entailment': 130899, 'contradiction': 130903}
Genre counts: {'government': 77350, 'telephone': 83348, 'fiction': 77348, 'travel': 77350, 'slate': 77306}
Duplicate rows: 22
Conflicting pairs: 83
Word lengths: {'premise': {'min': 1, 'mean': 19.81103228402198, 'median': 17.0, 'max': 382}, 'hypothesis': {'min': 1, 'mean': 9.972062785521846, 'median': 9.0, 'max': 69}}

validation_matched
Label counts: {'neutral': 3123, 'contradiction': 3213, 'entailment': 3479}
Genre counts: {'slate': 1955, 'government': 1945, 'telephone': 1966, 'travel': 1976, 'fiction': 1973}
Duplicate rows: 0
Conflicting pairs: 0
Word lengths: {'premise': {'min': 1, 'mean': 19.266123280692817, 'median': 16, 'max': 196}, 'hypothesis': {'min': 1, 'mean': 9.916352521650534, 'median': 9, 'max': 46}}

validation_mismatched
Label counts: {'contradiction': 3240, 'entailment': 3463, 'neutral': 3129}
Genre counts: {'letters': 1977, 'verbatim': 1946, 'facetoface': 1974, 'oup': 1961, 'ninee

## 2. Prepared subset and cross-validation folds

In [11]:
inspect_prepared_subset()

Prepared subset size: 50,001
Labels: {'entailment': 16667, 'contradiction': 16667, 'neutral': 16667}
Fold 0: 39,999 training, 10,002 validation
Fold 1: 39,999 training, 10,002 validation
Fold 2: 40,002 training, 9,999 validation
Fold 3: 40,002 training, 9,999 validation
Fold 4: 40,002 training, 9,999 validation


In [12]:
if (DATA_DIR / "manifest.json").exists():
    subset, folds = load_saved_folds(DATA_DIR)
    print("Subset size:", len(subset))
    for label, count in subset.to_pandas()["label"].value_counts().sort_index().items():
        print(f"{LABEL_NAMES[int(label)]}: {count:,} ({count / len(subset):.2%})")
else:
    print("Prepared data is not available. Run `uv run python run_data.py` first.")

Subset size: 50001
entailment: 16,667 (33.33%)
neutral: 16,667 (33.33%)
contradiction: 16,667 (33.33%)


## 3. Experiment metrics

In [14]:
inspect_experiment_metrics()

,model,fold,accuracy,macro_f1
9,logistic_indicator,fold-0,0.495801,0.495786
8,logistic_indicator,fold-1,0.497500,0.497718
5,logistic_indicator,fold-2,0.511151,0.511291
7,logistic_indicator,fold-3,0.502750,0.503132
6,logistic_indicator,fold-4,0.502150,0.502810
4,majority,fold-0,0.333333,0.166667
3,majority,fold-1,0.333333,0.166667
0,majority,fold-2,0.333333,0.166667
2,majority,fold-3,0.333333,0.166667
1,majority,fold-4,0.333333,0.166667


accuracy        macro_f1       
                       mean    std     mean    std
model                                             
logistic_indicator   0.5019  0.006   0.5021  0.006
majority             0.3333  0.000   0.1667  0.000

In [15]:
# Optional: inspect one saved predictions file and calculate a detailed report.
prediction_files = list(EXPERIMENT_DIR.glob("*/*/predictions.csv"))
if prediction_files:
    predictions = __import__("pandas").read_csv(prediction_files[0])
    print(prediction_files[0])
    display(predictions.head())
    display(predictions.groupby("genre").size().rename("examples"))
else:
    print("No prediction files found. Run `uv run python run_experiments.py` first.")

/Users/lindevansaane/Onedrive/Msc Artificial Intelligence /Natural Language Processing 1/Projects/NLP1-Natural_Language_Inference-Project/results/experiments/mnli-n50000-seed0/majority/fold-2/predictions.csv


,pairID,genre,gold_label,prediction,p_entailment,p_neutral,p_contradiction
0,94381c,travel,2,0,0.333333,0.333333,0.333333
1,94381n,travel,1,0,0.333333,0.333333,0.333333
2,94381e,travel,0,0,0.333333,0.333333,0.333333
3,57570c,fiction,2,0,0.333333,0.333333,0.333333
4,57570n,fiction,1,0,0.333333,0.333333,0.333333


genre
fiction       2091
government    1956
slate         1833
telephone     2154
travel        1965
Name: examples, dtype: int64

## 4. Add your own tests

Use this section for questions such as:

- Which labels are most often confused?
- Does performance differ by genre?
- Are longer examples harder?
- Does lexical overlap correlate with correct predictions?
- Does the model perform differently on matched and mismatched genres?

In [ ]:
run_small_sanity_tests()

# Example custom test: evaluate a small set of manually chosen predictions.
gold = [0, 1, 2, 0, 1, 2]
predictions = [0, 2, 2, 0, 1, 0]
evaluate_predictions(gold, predictions)